# Vertex analysis: ML runs and fiTQun

This notebook configures and runs the accompanying Python workflow from source files through selections, evaluation, and plots. Run it from the repository root. The implementation is shared with `examples/full/compare_vertex_samples.py`, so cuts and plot definitions stay consistent.

By default, only the truth-label selection is applied. The optional nhit and common fiducial-Wall cuts are explicit settings and, when enabled, are applied to every sample. This avoids silently comparing samples selected with different cuts. The Wall cut is off by default; enable it only when the analysis targets that fiducial region.

Resolution plots use the half-width of the central 16th–84th percentile interval for signed X/Y/Z/R residuals and the 68th percentile of 3D error magnitude. Bias plots show mean signed residual with standard error. Histogram bins are shared across samples. These are example choices; adjust them to the analysis definition, not just to match a local plotting convention.

## 1. Configure inputs and selections

In [ ]:
import importlib
import os
import sys
from pathlib import Path

REPOSITORY_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "examples" / "full" / "compare_vertex_samples.py").is_file()
    ),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Open this notebook from a checkout of the repository.")
os.chdir(REPOSITORY_ROOT)
examples_dir = REPOSITORY_ROOT / "examples" / "full"
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))

analysis = importlib.import_module("compare_vertex_samples")

analysis.ML_RUNS = [
    analysis.MLRunConfig(
        name="ML model A",
        source_hdf5=Path("data/test.h5"),
        predictions_path=Path("outputs/model_a/preds.npy"),
        targets_path=Path("outputs/model_a/targets.npy"),
        indices_path=Path(
            "outputs/model_a/indices.npy"
        ),  # use None if source row order matches predictions
        color="tab:blue",
    ),
    analysis.MLRunConfig(
        name="ML model B",
        source_hdf5=Path("data/test.h5"),
        predictions_path=Path("outputs/model_b/preds.npy"),
        targets_path=Path("outputs/model_b/targets.npy"),
        indices_path=Path("outputs/model_b/indices.npy"),
        color="tab:orange",
    ),
]

analysis.FITQUN_CONFIGS = [
    analysis.FitQunConfig(
        name="fiTQun electron fit",
        path=Path("data/common_multi_fitqun.hy"),
        color="black",
    ),
    # Add another config to compare an additional fiTQun file or fit.
]

analysis.SELECTED_LABEL = 1
analysis.MINIMUM_NHIT = None  # Set a threshold to apply the nhit cut.
analysis.APPLY_WALL_CUT = False
analysis.MINIMUM_WALL_CM = 100.0
analysis.MAXIMUM_WALL_CM = 1650.0
analysis.ENABLE_MATCHED_COMPARISON = False
analysis.ML_PREDICTIONS_NORMALIZED = False
analysis.OUTPUT_DIRECTORY = Path("outputs/vertex_comparison")

## 2. Load files, select events, and inspect cut flow

`load_samples()` loads each ML run and fiTQun configuration, calculates event-level quantities, and applies the same configured selection logic. When matched comparison is enabled, configure equivalent unique file/event ID keys in all inputs.

In [ ]:
ml_samples, fitqun_samples = analysis.load_samples()
samples = [*ml_samples, *fitqun_samples]

for sample in samples:
    print(f"\n{sample.name}: selected {sample.quantities.event_count:,} events")
    if sample.cut_flow:
        print(f"  Initial events: {sample.cut_flow[0].before:,}")
    for cut in sample.cut_flow:
        print(
            f"  {cut.name}: {cut.before:,} -> {cut.after:,} (removed {cut.removed:,})"
        )

## 3. Evaluate independent samples

Independent comparisons do not assume event-by-event correspondence. The workflow saves a shared-bin 3D-error distribution, Wall-dependent X/Y/Z/R/3D resolution, selected event counts, and signed-residual bias plots.

In [ ]:
analysis.plot_samples(samples, output_stem="independent_vertex_comparison")
print(f"Plots saved under {analysis.OUTPUT_DIRECTORY}")

## 4. Optional matched-event comparison

Matching aligns selected samples by unique `(file_id, event_id)` keys. It compares each ML/fiTQun pair on the common events but does not silently reinterpret the result as an ML-minus-fiTQun residual.

In [ ]:
if analysis.ENABLE_MATCHED_COMPARISON:
    matched_samples = analysis.make_matched_samples(ml_samples, fitqun_samples)
    analysis.plot_samples(matched_samples, output_stem="matched_vertex_comparison")
else:
    print(
        "Set ENABLE_MATCHED_COMPARISON and event-ID keys after "
        "confirming ID equivalence."
    )